In [1]:
%cd /home/parthgandhi/Projects/SwingBot/swingbot/src

/home/parthgandhi/Projects/SwingBot/swingbot/src


In [2]:
import logging
from datetime import datetime, timedelta

import polars as pl
import polars.selectors as cs

from config.ingestion.brokers import KiteConfig
from config.ingestion.data_sources import NSEConfig
from config.computation.compute import ComputeConfig
from config.computation.indicator import IndicatorConfig
from utils import setup_logger
from config.base import StorageConfig

logger = logging.getLogger(__name__)

setup_logger()

In [3]:
end_date = "2026-03-13"
save_path = StorageConfig().store_root(ComputeConfig.FOLDER_NAME, end_date)

# Scanner DataFrame

In [4]:
res = (
    pl.scan_csv(save_path / ComputeConfig.FILTER_RESULT_PATH)
    .select(
        [
            "symbol",
            "pct_gain_prev_1",
            "pct_gain_prev_5",
            "pct_gain_prev_21",
            "pct_gain_prev_63",
            "pct_gain_prev_126",
            "rvol_pct_50",
            "adr_pct_20",
            "adr_filter_flag",
            "pullback_filter_flag",
            "mid_down_streak",
            "near_ema_9",
            "near_ema_21",
            "near_sma_50",
            "macro_economic_sector",
            "sector",
            "industry",
            "basic_industry",
            "market_cap_cr",
        ]
    )
    .rename(
        {
            f"pct_gain_prev_{i}": f"{value}"
            for i, value in IndicatorConfig.LOOKBACK_RETURN_PCT.items()
        }
    )
    .collect()
)

res = (
    res.lazy()
    .rename({i: i.replace("_", " ").upper() for i in res.columns})
    .with_columns(cs.float().round(2))
    .collect()
)
res

SYMBOL,1D,1W,1M,3M,6M,RVOL PCT 50,ADR PCT 20,ADR FILTER FLAG,PULLBACK FILTER FLAG,MID DOWN STREAK,NEAR EMA 9,NEAR EMA 21,NEAR SMA 50,MACRO ECONOMIC SECTOR,SECTOR,INDUSTRY,BASIC INDUSTRY,MARKET CAP CR
str,f64,f64,f64,f64,f64,f64,f64,bool,bool,i64,bool,bool,bool,str,str,str,str,f64
"""AARTIIND""",-5.7,0.33,-9.93,17.86,8.19,100.0,3.96,true,true,1,true,true,false,"""Commodities""","""Chemicals""","""Chemicals & Petrochemicals""","""Specialty Chemicals""",15228.96
"""ABB""",-0.26,5.45,9.73,21.94,23.84,249.0,3.32,true,false,null,null,null,null,"""Industrials""","""Capital Goods""","""Electrical Equipment""","""Heavy Electrical Equipment""",135388.26
"""STYRENIX""",-2.04,8.95,2.11,-1.15,-23.28,221.0,2.84,false,true,1,true,true,true,null,null,null,null,null
"""GODREJAGRO""",-3.0,-4.41,-3.78,-3.84,-23.18,43.0,3.34,true,true,3,false,false,true,null,null,null,null,null
"""APCOTEXIND""",-1.48,-4.0,-4.81,-4.23,-13.79,450.0,2.83,false,true,2,true,false,false,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ATHERENERG""",-0.78,4.59,-1.57,9.06,30.18,122.0,3.99,true,true,0,true,true,true,null,null,null,null,null
"""PASHUPATI""",-0.77,-2.32,14.35,18.94,46.98,471.0,4.95,true,true,1,true,true,false,null,null,null,null,null
"""KNAGRI""",-0.48,-4.3,6.36,-10.76,-20.25,80.0,8.4,true,true,3,true,true,false,null,null,null,null,null


# Rolling RSS

In [25]:
stock_table_id = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=False
)
indices_table_id = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_INDICES, frequency="day", failed_tbl=False
)

query = f"""
    select *
    from {stock_table_id}
"""

stocks_df = pl.read_database_uri(query=query, uri=KiteConfig.DB_CONN)

query = f"""
    select *
    from {indices_table_id}
"""

indices_df = pl.read_database_uri(query=query, uri=KiteConfig.DB_CONN).filter(
    pl.col("symbol") == ComputeConfig.RS_INDEX
)

In [26]:
indices_df

symbol,timestamp,open,high,low,close,volume
str,datetime[ns],f64,f64,f64,f64,i64
"""NIFTY MIDSML 400""",2025-03-21 00:00:00,17521.5,17802.1,17477.15,17784.45,0
"""NIFTY MIDSML 400""",2025-03-24 00:00:00,17909.4,18052.2,17833.0,17975.1,0
"""NIFTY MIDSML 400""",2025-03-25 00:00:00,18104.9,18120.15,17694.4,17775.75,0
"""NIFTY MIDSML 400""",2025-03-26 00:00:00,17802.75,17880.6,17596.1,17614.75,0
"""NIFTY MIDSML 400""",2025-03-27 00:00:00,17550.0,17743.05,17484.35,17714.1,0
…,…,…,…,…,…,…
"""NIFTY MIDSML 400""",2026-03-10 00:00:00,18874.8,19010.0,18755.45,18963.9,0
"""NIFTY MIDSML 400""",2026-03-11 00:00:00,19019.3,19133.7,18763.35,18795.85,0
"""NIFTY MIDSML 400""",2026-03-12 00:00:00,18694.55,18865.8,18434.9,18742.0,0


In [70]:
def cal_atr(data: pl.DataFrame | pl.LazyFrame):

    res = (
        data.lazy()
        .with_columns(pl.col("timestamp").cast(pl.Date()))
        .with_columns(
            pl.col("close")
            .shift(1)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .alias("close_prev_1")
        )
        .with_columns(
            pl.max_horizontal(
                pl.col("high") - pl.col("low"),
                (pl.col("high") - pl.col("close_prev_1")).abs(),
                (pl.col("low") - pl.col("close_prev_1")).abs(),
            ).alias("true_range")
        )
        .with_columns(
            pl.col("true_range")
            .ewm_mean(alpha=2 / (n + 1))
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(2)
            .alias(f"atr_{n}")
            for n in [20]
        )
        .select("symbol", "timestamp", "close", "atr_20")
    )

    return res

In [71]:
index_atr = (
    cal_atr(data=indices_df)
    .rename({"atr_20": "index_atr_20"})
    .with_columns(
        pl.col("close")
        .shift(1)
        .over(partition_by="symbol", order_by="timestamp", descending=False)
        .alias("close_prev_1")
    )
    .with_columns((pl.col("close") - pl.col("close_prev_1")).alias("price_change"))
    .with_columns((pl.col("price_change") / pl.col("index_atr_20")).alias("spi"))
    .select("timestamp", "spi")
)
stocks_atr = cal_atr(data=stocks_df).rename({"atr_20": "stock_atr_20"}).drop("close")

In [ ]:
stocks_df.lazy().with_columns(pl.col("timestamp").cast(pl.Date())).with_columns(
    pl.col("close")
    .shift(1)
    .over(partition_by="symbol", order_by="timestamp", descending=False)
    .alias("close_prev_1")
).with_columns((pl.col("close") - pl.col("close_prev_1")).alias("price_change")).join(
    stocks_atr, on=["symbol", "timestamp"]
).join(index_atr, on="timestamp", how="left").with_columns(
    (pl.col("spi") * pl.col("stock_atr_20")).alias("expected_change")
).with_columns(
    (pl.col("price_change") - pl.col("expected_change")).alias("excess_move")
).with_columns(
    (pl.col("excess_move") / pl.col("stock_atr_20")).alias("rss_20")
).with_columns(
    pl.col("rss_20")
    .rolling_mean(20)
    .over(partition_by="symbol", order_by="timestamp", descending=False)
    .alias("rolling RSS")
).select("symbol", "timestamp", "rss_20", "rolling RSS").with_columns(
    pl.col("rolling RSS")
    .rank(method="dense")
    .over("timestamp", descending=False)
    .alias("rank")
).with_columns(
    ((pl.col("rank") - 1) / (pl.len().over("timestamp")) * 100).alias("rss_score")
).collect().filter(pl.col("timestamp") == datetime(2026, 3, 16)).filter(
    pl.col("rss_score").is_not_null()
).sort("rss_score", descending=True).with_columns(cs.float().round(2)).write_csv("rss_test.csv")

In [69]:
index_atr.with_columns(
    (pl.col("price_change") / pl.col("index_atr_20")).alias("spi")
).select("timestamp", "spi").collect()

timestamp,spi
date,f64
2025-03-21,null
2025-03-24,0.646446
2025-03-25,-0.581263
2025-03-26,-0.493744
2025-03-27,0.320711
…,…
2026-03-10,0.947641
2026-03-11,-0.47857
2026-03-12,-0.150109
